In [1]:
import pandas as pd
import networkx as nx 
from matplotlib import pyplot as plt
import numpy as np

In [2]:
edge_list = pd.read_csv(r"D:\commo\code\4_kumu_struct\edge_list.csv")
node1 = pd.read_csv(r"D:\commo\code\4_kumu_struct\node_list1.csv")
node2 = pd.read_csv(r"D:\commo\code\4_kumu_struct\node_list2.csv")
node3 = pd.read_csv(r"D:\commo\code\4_kumu_struct\node_list3.csv")
node1.head()

,Label,Type,Description,Tier,Commodity Type,Commodity Focus,Legal Name / ACRA UEN,HQ Country,Estimated Revenue,Headcount (Global),Trade Volume (Est.),Ownership Structure,Exchange,APAC offices
0,trafigura group,"trader, lender",Latest acquisition (after Sep 2025): Greenergy...,3,multi (>2 types),oil | petroleum | natural gas | LNG | metals |...,TRAFIGURA GROUP PTE. LTD. / 201017488D (record...,Singapore,Trafigura Group: 240.3 USD bn FY2025 (trafigur...,5011 (linkedin),approx. 462 million metric tons (2025 annual r...,private,NaN,"10 Collyer Quay, #29-01/05, Ocean Financial Ce..."
1,vitol asia,trader,"Its largest operations are in Geneva, Houston,...",3,multi (>2 types),crude oil | petroleum | LNG | natural gas | bi...,VITOL ASIA PTE LTD. / 199001917Z (recordowl),Switzerland,Turnover of $343 billion FY2025 (company website),1990 (linkedin),total energy trade volume: 605 million tonnes ...,private,NaN,128 BEACH ROAD #28-01 GUOCO MIDTOWN OFFICE SIN...
2,mercuria holdings,others,Mercuria Holdings (Singapore) Pte. Ltd. is a g...,3,multi (>2 types),energy | metals | soft/agricultural,MERCURIA HOLDINGS (SINGAPORE) PTE. LTD. / 2018...,Switzerland,$618.7 Million (zoom.info),1380 (linkedin),6M+ barrels of oil equivalent (BOE) per day (c...,private,NaN,"12 Marina View, #26-01, Asia Square Tower 2, S..."
3,gunvor singapore,trader,one of the world's largest independent commodi...,3,energy,crude oil | refined petroleum products | natur...,GUNVOR SINGAPORE PTE. LTD. / 200606959K (acra ...,Switzerland,US $144 billion (company website),1035 (linkedin),253 million MT (company website),private,NaN,"128 Beach Road, #29-01, Guoco Midtown Office, ..."
4,louis dreyfus company asia,trader,"LDC is purely agricultural, no energy commodit...",3,soft/agricultural,grains | oilseeds | coffee | cotton | sugar | ...,LOUIS DREYFUS COMPANY ASIA PTE. LTD. / 1993065...,Netherlands,Net Sales: US$ 53.2bn FY2025 (2025.12 LDC - In...,"13,359 (linkedin)",~100 million tonnes of products shipped annual...,private,NaN,12 MARINA BOULEVARD #33-03 MARINA BAY FINANCIA...


In [3]:
node1.rename(columns={
    'Label':'label',
    'Type':'role',
    'Description':'description',
    'Tier':'tier',
    'Commodity Type':'commodity_type',
    'Commodity Focus':'commodity_focus',
    'Legal Name / ACRA UEN':'acra_uen',
    'HQ Country':'hq_country',
    'Estimated Revenue':'est_revenue',
    'Headcount (Global)':'global_headcount',
    'Trade Volume (Est.)':'est_trade_volume',
    'Ownership Structure':"ownership_struct",
    'Exchange':'exchange',
    'APAC offices':'sg_address'
}, inplace=True)
node2.rename(columns={
    'Label':'label',
    'Type':'role',
    'Description':'description',
    'Scope':'scope',
    'Office Location(s) in APAC':'sg_address',
    'Practice Areas':'practice_areas',
    'Known Client Base':'known_clients'
}, inplace=True)
node3.rename(columns={
    'Label':'label',
    'Type':'role',
    'Description':'description',
    'Financier Type':'financieir_type',
    'Commodity Type':'commodity_type',
    'Geographic Reach':'geographic_reach',
    'Known Clients':'known_clients'
}, inplace=True)

In [4]:
# create undirected graph
UG = nx.Graph()
# add all nodes
# add all nodes
records1 = node1.set_index("label").to_dict("index")
records2 = node2.set_index("label").to_dict("index")
records3 = node3.set_index("label").to_dict("index")
UG.add_nodes_from(records1.items())
UG.add_nodes_from(records2.items())
UG.add_nodes_from(records3.items())

# add edges
edges = [(row.From, row.To, {'type':row.Type}) for row in edge_list[['From','To','Type']].itertuples()]
UG.add_edges_from(edges)
# UG['trafigura group']['trafigura carbon trading'] > {'type':'owns'}

def remove_isolated_nodes_undirected(G):
    G = G.copy()
    isolated = [n for n in G.nodes() if G.degree(n) == 0]
    G.remove_nodes_from(isolated)
    return G

UG = remove_isolated_nodes_undirected(UG)
print(UG.number_of_nodes(), "nodes remaining")

# community detection, run five times with different random seeds
run1 = nx.community.louvain_communities(UG, seed=123) # visualize
run2 = nx.community.louvain_communities(UG, seed=234)
run3 = nx.community.louvain_communities(UG, seed=345)
run4 = nx.community.louvain_communities(UG, seed=456)
run5 = nx.community.louvain_communities(UG, seed=567)

runs = {
    123: run1,
    234: run2,
    345: run3,
    456: run4,
    567: run5,
}

print(f"{'Seed':<8}{'# Communities':<16}{'Modularity':<12}")
for seed, run in runs.items():
    mod = nx.community.modularity(UG, run)
    print(f"{seed:<8}{len(run):<16}{mod:.4f}")

best_run = run3  # seed 345
for i, community in enumerate(best_run):
    print(f"\nCommunity {i} ({len(community)} nodes):")
    for node in community:
        print(" ", node)

164 nodes remaining
Seed    # Communities   Modularity  
123     28              0.7661
234     26              0.7666
345     28              0.7672
456     27              0.7654
567     28              0.7661

Community 0 (6 nodes):
  seatrium
  virtus law llp
  standard chartered
  standard chartered bank singapore scb
  shook lin & bok llp
  marubeni corporation

Community 1 (2 nodes):
  mitsui & co energy trading singapore (mets)
  mitsui & co

Community 2 (3 nodes):
  shell singapore
  baker mckenzie
  shell eastern trading

Community 3 (5 nodes):
  olam group
  olam food ingredients
  stephenson harwood
  olam agri holdings
  icici bank singapore branch

Community 4 (11 nodes):
  asialegal llc
  dbs bank
  louis dreyfus company asia
  louis dreyfus company
  agrocorp international
  rio tinto
  wongpartnership
  norton rose fulbright
  pdlegal llc
  first resources
  first sponsor group

Community 5 (9 nodes):
  cargill
  helmsman llc
  united overseas bank uob
  barramundi gro

In [72]:
# create a directed graph
DG = nx.DiGraph()

# add all nodes
records1 = node1.set_index('label').to_dict('index')
records2 = node2.set_index("label").to_dict("index")
records3 = node3.set_index("label").to_dict("index")
DG.add_nodes_from(records1.items())
DG.add_nodes_from(records2.items())
DG.add_nodes_from(records3.items())

# add edges
edges = [(row.From, row.To, {'type':row.Type}) for row in edge_list[['From','To','Type']].itertuples()]
DG.add_edges_from(edges) # full edge list

# number of nodes and edges
no_nodes = nx.number_of_nodes(DG)
no_edges = nx.number_of_edges(DG)
print('Original number of nodes and edges: ',no_nodes, no_edges)

# density
print()
density_dg = nx.density(DG) # visualize
print('Density of directed graph: ', density_dg)

# degree-centrality, in-degree, out-degree
print()
degree_centrality = nx.degree_centrality(DG) # visualize
print('Nodes with highest degree centrality:')
for label, score in sorted(list(degree_centrality.items())[:10], key=lambda x: x[1], reverse=True):
    print(label, score)

print()
in_degree_centrality = nx.in_degree_centrality(DG) # visualize
print('Nodes with highest in-degree centrality:')
for label, score in sorted(list(in_degree_centrality.items())[:10], key=lambda x: x[1], reverse=True):
    print(label, score)

print()
out_degree_centrality = nx.out_degree_centrality(DG) # visualize
print('Nodes with highest out-degree centrality:')
for label, score in sorted(list(out_degree_centrality.items())[:10], key=lambda x: x[1], reverse=True):
    print(label, score)

# betweenness centrality
print()
betweenness_centrality = nx.betweenness_centrality(DG) # visualize
print('Nodes with highest betweenness centrality:')
for label, score in sorted(list(betweenness_centrality.items())[:10], key=lambda x: x[1], reverse=True):
    print(label, score)

# pagerank v2 using networkx built-in function
DG_copy = DG.copy()
print()
print("Removing isolated nodes..")
all_nodes = list(DG_copy.nodes())
# DG.edges({node}) shows all the edges, len(DG.edges({node})) shows the no of edges that node has
for node in all_nodes:
    if DG_copy.in_degree(node) == 0 and DG_copy.out_degree(node) == 0:
        DG_copy.remove_node(node)
print('Remaining nodes and edges after removing isolated nodes:', DG_copy.number_of_nodes(), DG_copy.number_of_edges())  # 296 nodes remaining
# labelling nodes as integers
print("Relabelling nodes to integers...")
print()
# relabel nodes as integers
n_unique_nodes = len(set(DG_copy.nodes()))  # 296 nodes total
# create dict{'label':int}
node2int = dict(zip(set(DG_copy.nodes()), range(n_unique_nodes)))  # {'cofco international': 0,'hyundai corporation singapore': 1, etc}
# create an opp dict
int2node = {v: k for k, v in node2int.items()}  # {0: 'cofco international',1: 'hyundai corporation singapore', etc}
# relabel the directed graph with the node2int dict
DG_copy = nx.relabel_nodes(DG_copy, node2int)
pagerank_score = nx.pagerank(DG_copy, alpha=0.85)
ranked = sorted(pagerank_score.items(), key=lambda x: x[1], reverse=True)
print('Nodes with highest pagerank score:')
for idx, score in ranked[:10]:
    print(int2node[idx], score)

# constraints to get structural holes
print()
constraints = nx.constraint(DG_copy)
constraints_ranked = sorted(constraints.items(), key=lambda x: x[1], reverse=False)
print('Nodes with lowest constraint score:')
for idx, score in constraints_ranked[:10]:
    print(int2node[idx], score)

Original number of nodes and edges:  296 169

Density of directed graph:  0.001935409986257444

Nodes with highest degree centrality:
trafigura group 0.06440677966101695
louis dreyfus company asia 0.013559322033898305
wilmar international 0.013559322033898305
olam group 0.010169491525423728
vitol asia 0.003389830508474576
mercuria holdings 0.003389830508474576
gunvor singapore 0.003389830508474576
cofco international singapore 0.003389830508474576
cargill asia pacific holdings 0.003389830508474576
glencore singapore 0.003389830508474576

Nodes with highest in-degree centrality:
trafigura group 0.04406779661016949
louis dreyfus company asia 0.013559322033898305
wilmar international 0.010169491525423728
vitol asia 0.003389830508474576
mercuria holdings 0.003389830508474576
gunvor singapore 0.003389830508474576
olam group 0.003389830508474576
cofco international singapore 0.003389830508474576
cargill asia pacific holdings 0.003389830508474576
glencore singapore 0.003389830508474576

Nodes